# Pandas: Solutions

Instructor copy. Run the setup cell first, then any solution below.

In [ ]:
# SETUP: recreate the data files
import os
import pandas as pd
import numpy as np

with open("airtravel.csv", "w", encoding="utf-8") as f:
    f.write('"Month", "1958", "1959", "1960"\n')
    for row in [
        ("JAN", 340, 360, 417), ("FEB", 318, 342, 391), ("MAR", 362, 406, 419),
        ("APR", 348, 396, 461), ("MAY", 363, 420, 472), ("JUN", 435, 472, 535),
        ("JUL", 491, 548, 622), ("AUG", 505, 559, 606), ("SEP", 404, 463, 508),
        ("OCT", 359, 407, 461), ("NOV", 310, 362, 390), ("DEC", 337, 405, 432),
    ]:
        f.write(f'"{row[0]}",  {row[1]},  {row[2]},  {row[3]}\n')

with open("orders.csv", "w", encoding="utf-8") as f:
    f.write("order_id,customer_id,item,category,quantity,unit_price,order_date,country,status\n")
    f.write("1001,101,Laptop Pro,Electronics,1,450000,2026-01-15,NG,completed\n")
    f.write("1002,102,Phone X,Electronics,2,180000,2026-01-18,GH,completed\n")
    f.write("1003,101,Wireless Mouse,Accessories,3,8000,2026-02-03,NG,completed\n")
    f.write("1004,103,Laptop Pro,Electronics,1,520000,2026-02-11,NG,pending\n")
    f.write("1005,104,Monitor 27,Electronics,2,95000,2026-02-20,KE,completed\n")
    f.write("1006,102,Wireless Mouse,Accessories,1,7500,2026-03-02,GH,cancelled\n")
    f.write("1007,103,Keyboard,Accessories,2,15000,2026-03-14,NG,completed\n")
    f.write("1008,105,Phone X,Electronics,1,,2026-03-19,KE,completed\n")
    f.write("1009,101,Monitor 27,Electronics,1,95000,2026-04-01,NG,completed\n")
    f.write("1010,106,Headset,Accessories,4,22000,2026-04-08,GH,completed\n")
    f.write("1011,102,Laptop Pro,Electronics,1,450000,,NG,pending\n")
    f.write("1012,999,Keyboard,Accessories,1,15000,2026-04-22,NG,completed\n")
    f.write("1003,101,Wireless Mouse,Accessories,3,8000,2026-02-03,NG,completed\n")

with open("customers.csv", "w", encoding="utf-8") as f:
    f.write("customer_id,name,segment,signup_date,country\n")
    f.write("101,Ahmed Oladapo,enterprise,2025-06-01,NG\n")
    f.write("102,Lahla Mensah,smb,2025-08-14,GH\n")
    f.write("103,Judith Eze,enterprise,2025-09-30,NG\n")
    f.write("104,Uche Okafor,smb,2026-01-05,KE\n")
    f.write("105,Grace Wanjiru,smb,2026-01-20,KE\n")
    f.write("107,Samuel Adeyemi,enterprise,2026-02-11,NG\n")

with open("orders_may.csv", "w", encoding="utf-8") as f:
    f.write("order_id,customer_id,item,category,quantity,unit_price,order_date,country,status\n")
    f.write("1013,104,Laptop Pro,Electronics,1,450000,2026-05-04,KE,completed\n")
    f.write("1014,101,Headset,Accessories,2,22000,2026-05-12,NG,completed\n")

print("setup done, pandas", pd.__version__)

## Inspecting and loading: tasks

In [ ]:
# Task 1
air = pd.read_csv("airtravel.csv", skipinitialspace=True)
print(list(air.columns))
print(air.shape)
print(air.dtypes)

In [ ]:
# Task 2
peek = pd.read_csv("orders.csv", usecols=["order_id", "item", "quantity"], nrows=5)
peek

In [ ]:
# Task 3
orders = pd.read_csv("orders.csv", parse_dates=["order_date"])
print(orders.describe())
# unit_price has the widest spread: roughly 7,500 to 520,000.

In [ ]:
# DE Task 4: a reusable profile function
def profile(df, name="dataframe"):
    """Print the standard first-look summary of any DataFrame."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "distinct": df.nunique(),
    })
    print(summary)
    print()
    print(f"fully duplicated rows: {df.duplicated().sum()}")

profile(orders, "orders.csv")

In [ ]:
# DE Task 5: snake_case column names in one chain
clean = pd.read_csv("orders.csv")
clean.columns = (
    clean.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)
print(list(clean.columns))

In [ ]:
# DE Task 6
print(orders["status"].value_counts())
print()
print(orders["country"].value_counts())
print()
not_completed = (orders["status"] != "completed").mean() * 100
print(f"{not_completed:.1f}% of orders are not completed")

## Selecting and filtering: tasks

In [ ]:
# Task 1
print("iloc[0:4] gives 4 rows (stop excluded):")
print(orders.iloc[0:4][["order_id", "item"]])
print()
print("loc[0:4] gives 5 rows (stop included):")
print(orders.loc[0:4, ["order_id", "item"]])

In [ ]:
# Task 2
print(orders[orders["quantity"] > 1][["order_id", "item", "quantity"]])

In [ ]:
# Task 3
result = orders[(orders["category"] == "Accessories") & (orders["country"] == "NG")]
print(result[["order_id", "item", "category", "country"]])

In [ ]:
# Task 4
print(orders.query("category == 'Accessories' and country == 'NG'")
      [["order_id", "item", "category", "country"]])

In [ ]:
# DE Task 5
result = orders[~(orders["status"] == "completed") & (orders["country"] != "NG")]
print(result[["order_id", "status", "country"]])

In [ ]:
# DE Task 6
laptops = orders[orders["item"].str.contains("Laptop", na=False)].copy()
laptops["revenue"] = laptops["quantity"] * laptops["unit_price"]
print(laptops[["order_id", "item", "quantity", "unit_price", "revenue"]])
print(f"total laptop revenue: {laptops['revenue'].sum():,.0f}")

In [ ]:
# DE Task 7
ng = orders[orders["country"] == "NG"].copy()
ng["vat"] = ng["unit_price"] * 0.075

print(ng[["order_id", "unit_price", "vat"]].head())
print()
print(f"original has a vat column? {'vat' in orders.columns}")

## Missing data and types: tasks

In [ ]:
# Task 1
print(pd.DataFrame({
    "nulls": orders.isna().sum(),
    "null_pct": (orders.isna().mean() * 100).round(1),
}))

In [ ]:
# Task 2
print(orders[orders.isna().any(axis=1)])

In [ ]:
# Task 3
filled_zero = orders.copy()
filled_zero["unit_price"] = filled_zero["unit_price"].fillna(0)

filled_mean = orders.copy()
filled_mean["unit_price"] = filled_mean["unit_price"].fillna(orders["unit_price"].mean())

print(filled_zero[["order_id", "unit_price"]].iloc[[7]])
print(filled_mean[["order_id", "unit_price"]].iloc[[7]])

# For a revenue report, neither. Filling with 0 understates revenue and filling with
# the mean invents money. Leave it null, exclude the row, and say so in the output.

In [ ]:
# Task 4
dated = pd.read_csv("orders.csv")
dated["order_date"] = pd.to_datetime(dated["order_date"], errors="coerce")
dated["year"] = dated["order_date"].dt.year
dated["month"] = dated["order_date"].dt.month
dated["partition"] = dated["order_date"].dt.strftime("%Y-%m")

print(dated[["order_id", "order_date", "year", "month", "partition"]].head(6))

In [ ]:
# DE Task 5
def to_number(series, name="column"):
    """Convert to numeric with coercion, reporting how many values became null."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    numeric = pd.to_numeric(cleaned, errors="coerce")
    before = series.isna().sum()
    after = numeric.isna().sum()
    print(f"{name}: coerced {after - before} value(s) to null")
    return numeric

messy = pd.Series(["1000", "2,500", "N/A", "", "750"])
print(to_number(messy, "amount").tolist())

In [ ]:
# DE Task 6
dupes = orders[orders["order_id"].duplicated(keep=False)]
print("every copy of each duplicated order_id:")
print(dupes[["order_id", "item", "order_date"]])

deduped = orders.drop_duplicates(subset=["order_id"], keep="first")
print()
print(f"{len(orders)} -> {len(deduped)}, dropped {len(orders) - len(deduped)}")

In [ ]:
# DE Task 7
def type_check(df, expected):
    """Compare each column's actual dtype against what you expected."""
    checkers = {
        "int": pd.api.types.is_integer_dtype,
        "float": pd.api.types.is_float_dtype,
        "numeric": pd.api.types.is_numeric_dtype,
        "datetime": pd.api.types.is_datetime64_any_dtype,
        "string": lambda s: pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s),
    }
    for column, wanted in expected.items():
        if column not in df.columns:
            print(f"{column:<12} MISSING from the dataframe")
            continue
        ok = checkers[wanted](df[column])
        mark = "ok " if ok else "NO "
        print(f"{mark} {column:<12} expected {wanted:<9} actual {df[column].dtype}")

type_check(orders, {
    "order_id": "int",
    "unit_price": "float",
    "order_date": "datetime",
    "item": "string",
})

## Grouping and aggregating: tasks

In [ ]:
# Task 1
orders = pd.read_csv("orders.csv", parse_dates=["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders[["order_id", "quantity", "unit_price", "revenue"]].head()

In [ ]:
# Task 2
print(orders.groupby("category")["revenue"].sum().sort_values(ascending=False))

In [ ]:
# Task 3
print(orders.groupby("country").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_revenue=("revenue", "mean"),
).round(0).reset_index())

In [ ]:
# Task 4
print(orders["status"].value_counts())
print()
print(orders.groupby("status")["order_id"].count())

In [ ]:
# DE Task 5
by_customer = (
    orders.groupby("customer_id")["revenue"].sum()
    .reset_index()
    .query("revenue > 100000")
    .sort_values("revenue", ascending=False)
)
by_customer

In [ ]:
# DE Task 6
by_month = orders.groupby(
    orders["order_date"].dt.strftime("%Y-%m"), dropna=False
).agg(
    orders=("order_id", "count"),
    customers=("customer_id", "nunique"),
    revenue=("revenue", "sum"),
).reset_index(names="month")

by_month

In [ ]:
# DE Task 7
orders["category_total"] = orders.groupby("category")["revenue"].transform("sum")
orders["pct_of_category"] = (orders["revenue"] / orders["category_total"] * 100).round(1)

orders[["order_id", "category", "revenue", "category_total", "pct_of_category"]].head(8)

In [ ]:
# DE Task 8
pivot = orders.pivot_table(
    values="revenue", index="country", columns="status",
    aggfunc="sum", fill_value=0, margins=True,
)
pivot

## Merging and concatenating: tasks

In [ ]:
# Task 1
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])
joined = orders.merge(customers, on="customer_id", how="left")
print(joined[["order_id", "item", "name"]].head(8))

In [ ]:
# Task 2
inner = orders.merge(customers, on="customer_id", how="inner")
print(f"left join: {len(orders.merge(customers, on='customer_id', how='left'))} rows")
print(f"inner join: {len(inner)} rows")

lost = set(orders["order_id"]) - set(inner["order_id"])
print(f"lost {len(lost)} order(s): {sorted(lost)}")
# The inner join silently drops orders from customers 106 and 999.

In [ ]:
# DE Task 3
merged = orders.merge(customers, on="customer_id", how="outer", indicator=True)

print(merged["_merge"].value_counts())
print()
print("orphan orders:")
print(merged[merged["_merge"] == "left_only"][["order_id", "customer_id", "item"]])
print()
print("customers with no orders:")
print(merged[merged["_merge"] == "right_only"][["customer_id", "name", "segment"]])

In [ ]:
# DE Task 4
safe = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
print(f"validated join: {len(safe)} rows, matches {len(orders)} orders")

broken = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
try:
    orders.merge(broken, on="customer_id", how="left", validate="many_to_one")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:80]}")

unchecked = orders.merge(broken, on="customer_id", how="left")
print(f"without validate, {len(orders)} orders became {len(unchecked)} rows")

In [ ]:
# DE Task 5
may = pd.read_csv("orders_may.csv", parse_dates=["order_date"])
may["revenue"] = may["quantity"] * may["unit_price"]

combined = pd.concat([orders, may], ignore_index=True)
combined["month"] = combined["order_date"].dt.strftime("%Y-%m")

print(f"{len(orders)} + {len(may)} = {len(combined)}")
print(combined.groupby("month")["revenue"].sum())

In [ ]:
# DE Task 6
suffixed = orders.merge(customers, on="customer_id", how="left",
                        suffixes=("_order", "_customer"))

mismatch = suffixed[
    suffixed["country_customer"].notna()
    & (suffixed["country_order"] != suffixed["country_customer"])
]
print(f"{len(mismatch)} mismatch(es)")
print(mismatch[["order_id", "name", "country_order", "country_customer"]])

In [ ]:
# DE Task 7
def join_report(left, right, key):
    """Outer merge with an indicator, reporting match rates on both sides."""
    merged = left.merge(right, on=key, how="outer", indicator=True)
    counts = merged["_merge"].value_counts()
    matched = counts.get("both", 0)
    left_only = counts.get("left_only", 0)
    right_only = counts.get("right_only", 0)

    print(f"key: {key}")
    print(f"  matched:    {matched}")
    print(f"  left only:  {left_only}")
    print(f"  right only: {right_only}")
    if len(left):
        print(f"  left match rate:  {matched / (matched + left_only) * 100:.1f}%")
    if len(right):
        print(f"  right match rate: {matched / (matched + right_only) * 100:.1f}%")
    return merged

_ = join_report(orders, customers, "customer_id")

## Transforming: tasks

In [ ]:
# Task 1
orders = pd.read_csv("orders.csv", parse_dates=["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders["vat"] = orders["revenue"] * 0.075
orders["total"] = orders["revenue"] + orders["vat"]

orders[["order_id", "revenue", "vat", "total"]].head()

In [ ]:
# Task 2
import numpy as np
orders["size"] = np.where(orders["revenue"] > 200000, "large", "small")
print(orders[["order_id", "revenue", "size"]].head(8))

In [ ]:
# Task 3
conditions = [
    orders["unit_price"].isna(),
    orders["unit_price"] > 400000,
    orders["unit_price"] > 50000,
]
orders["tier"] = np.select(conditions, ["unknown", "premium", "mid"], default="budget")
print(orders[["order_id", "unit_price", "tier"]])

In [ ]:
# Task 4
labels = {"completed": "Done", "pending": "In progress", "cancelled": "Cancelled"}
orders["status_label"] = orders["status"].map(labels)
print(orders[["status", "status_label"]].drop_duplicates())

In [ ]:
# DE Task 5: one chain
report = (
    pd.read_csv("orders.csv", parse_dates=["order_date"])
    .assign(
        revenue=lambda d: d["quantity"] * d["unit_price"],
        vat=lambda d: d["revenue"] * 0.075,
        total=lambda d: d["revenue"] + d["vat"],
        size=lambda d: np.where(d["revenue"] > 200000, "large", "small"),
        tier=lambda d: np.select(
            [d["unit_price"].isna(), d["unit_price"] > 400000, d["unit_price"] > 50000],
            ["unknown", "premium", "mid"],
            default="budget",
        ),
    )
    .query("status == 'completed'")
    .sort_values("revenue", ascending=False)
)

report[["order_id", "item", "revenue", "total", "size", "tier"]]

In [ ]:
# DE Task 6
tidy = pd.read_csv("orders.csv", parse_dates=["order_date"])
tidy.columns = tidy.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
tidy = tidy.drop(columns=["quantity", "status"])

first = ["order_id", "order_date"]
tidy = tidy[first + [c for c in tidy.columns if c not in first]]
print(list(tidy.columns))
tidy.head(3)

In [ ]:
# DE Task 7: timing
import time

big = pd.concat([orders] * 5000, ignore_index=True)
print(f"{len(big):,} rows")

start = time.perf_counter()
_ = big["quantity"] * big["unit_price"]
t_vec = time.perf_counter() - start

start = time.perf_counter()
_ = big.apply(lambda r: r["quantity"] * r["unit_price"], axis=1)
t_apply = time.perf_counter() - start

print(f"vectorised: {t_vec * 1000:.1f} ms")
print(f"apply:      {t_apply * 1000:.1f} ms")
print(f"apply is {t_apply / t_vec:.0f}x slower")

## Mini project: full worked solution

In [ ]:
import pandas as pd, numpy as np, os, shutil

# 1. PROFILE
raw = pd.read_csv("orders.csv")

def profile(df, name):
    print(f"=== {name} ===")
    print(f"shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
    }))
    print(f"fully duplicated rows: {df.duplicated().sum()}")
    print(f"duplicate order_ids:   {df['order_id'].duplicated().sum()}")

profile(raw, "raw orders")

In [ ]:
# 2. CLEAN
def clean(df):
    """Parse types, drop duplicates, return (clean_df, rejected_df)."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

    rejected = []

    dates = pd.to_datetime(df["order_date"], errors="coerce")
    bad_dates = dates.isna() & df["order_date"].notna()
    prices = pd.to_numeric(df["unit_price"], errors="coerce")
    bad_prices = prices.isna() & df["unit_price"].notna()

    print(f"unparseable dates:  {bad_dates.sum()}")
    print(f"missing dates:      {dates.isna().sum()}")
    print(f"unparseable prices: {bad_prices.sum()}")
    print(f"missing prices:     {prices.isna().sum()}")

    df["order_date"] = dates
    df["unit_price"] = prices

    dupes = df[df["order_id"].duplicated(keep="first")].copy()
    dupes["reason"] = "duplicate order_id"
    rejected.append(dupes)
    df = df.drop_duplicates(subset=["order_id"], keep="first")
    print(f"dropped {len(dupes)} duplicate row(s)")

    no_price = df[df["unit_price"].isna()].copy()
    no_price["reason"] = "missing unit_price"
    rejected.append(no_price)
    df = df[df["unit_price"].notna()]

    rejected_df = pd.concat(rejected, ignore_index=True) if rejected else pd.DataFrame()
    return df.reset_index(drop=True), rejected_df


clean_df, rejected_df = clean(raw)
print(f"\nkept {len(clean_df)}, rejected {len(rejected_df)}")

In [ ]:
# 3. ENRICH
def enrich(df):
    """Add the derived columns a report needs."""
    return df.assign(
        revenue=lambda d: d["quantity"] * d["unit_price"],
        vat=lambda d: d["revenue"] * 0.075,
        total=lambda d: d["revenue"] + d["vat"],
        month=lambda d: d["order_date"].dt.strftime("%Y-%m"),
        tier=lambda d: np.select(
            [d["unit_price"].isna(), d["unit_price"] > 400000, d["unit_price"] > 50000],
            ["unknown", "premium", "mid"],
            default="budget",
        ),
    )

enriched = enrich(clean_df)
enriched[["order_id", "revenue", "vat", "total", "month", "tier"]].head()

In [ ]:
# 4. JOIN
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])

joined = enriched.merge(
    customers, on="customer_id", how="left",
    validate="many_to_one", indicator=True, suffixes=("", "_customer"),
)

orphans = joined[joined["_merge"] == "left_only"]
print(f"rows before join: {len(enriched)}, after: {len(joined)}")
print(f"orphan orders (no matching customer): {len(orphans)}")
print(orphans[["order_id", "customer_id", "item"]])

In [ ]:
# 5. AGGREGATE
by_country = joined.groupby("country").agg(
    orders=("order_id", "count"),
    customers=("customer_id", "nunique"),
    revenue=("revenue", "sum"),
).reset_index().sort_values("revenue", ascending=False)

by_cat_month = joined.groupby(["category", "month"])["revenue"].sum().reset_index()

by_segment = joined.dropna(subset=["segment"]).groupby("segment").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    avg_order_value=("revenue", "mean"),
).round(0).reset_index()

print(by_country)
print()
print(by_cat_month)
print()
print(by_segment)

In [ ]:
# 6. DATA QUALITY REPORT
completed = joined[joined["status"] == "completed"]

print("=== ORDERS PIPELINE REPORT ===")
print(f"Rows in:            {len(raw)}")
print(f"Rows kept:          {len(clean_df)}")
print(f"Rows rejected:      {len(rejected_df)}")
for _, row in rejected_df.iterrows():
    print(f"   -> order {row['order_id']}: {row['reason']}")
print(f"Rejection rate:     {len(rejected_df) / len(raw) * 100:.1f}%")
print()
print("Null rate after cleaning:")
for column, pct in (joined.isna().mean() * 100).round(1).items():
    if pct > 0:
        print(f"   {column}: {pct}%")
print()
print(f"Orphan orders:      {len(orphans)} -> {list(orphans['order_id'])}")
print(f"Completed orders:   {len(completed)} of {len(joined)}")
print(f"Revenue (completed only): {completed['revenue'].sum():,.0f}")
print(f"VAT (completed only):     {completed['vat'].sum():,.0f}")

In [ ]:
# 7. EXPORT
export_cols = [
    "order_id", "customer_id", "name", "segment", "item", "category",
    "quantity", "unit_price", "revenue", "vat", "total",
    "order_date", "month", "country", "status", "tier",
]
joined[export_cols].to_csv("clean_orders.csv", index=False)
by_country.to_csv("summary_by_country.csv", index=False)
rejected_df.to_csv("rejected_orders.csv", index=False)

for name in ["clean_orders.csv", "summary_by_country.csv", "rejected_orders.csv"]:
    print(f"{name:<26} {os.path.getsize(name):>6} bytes")

In [ ]:
# Stretch 8: partitioned write, then verify the round trip
import glob

partitioned = joined.dropna(subset=["order_date"]).copy()
for month, group in partitioned.groupby("month"):
    folder = os.path.join("output", f"month={month}")
    os.makedirs(folder, exist_ok=True)
    group[export_cols].to_csv(os.path.join(folder, "orders.csv"), index=False)

frames = [pd.read_csv(p) for p in sorted(glob.glob("output/**/*.csv", recursive=True))]
back = pd.concat(frames, ignore_index=True)

print(f"wrote {len(frames)} partition(s)")
print(f"rows out: {len(partitioned)}, rows back: {len(back)}")
print(f"revenue out: {partitioned['revenue'].sum():,.0f}, back: {back['revenue'].sum():,.0f}")
print("round trip verified" if len(back) == len(partitioned) else "MISMATCH, investigate")

In [ ]:
# Stretch 9 and 10
joined["country_revenue"] = joined.groupby("country")["revenue"].transform("sum")
joined["pct_of_country_revenue"] = (
    joined["revenue"] / joined["country_revenue"] * 100
).round(1)

print(joined[["order_id", "country", "revenue", "pct_of_country_revenue"]].head(8))
print()
print(joined.pivot_table(
    values="revenue", index="country", columns="month",
    aggfunc="sum", fill_value=0, margins=True,
))

In [ ]:
# Stretch 11: the whole thing as one schedulable function
def run(orders_path="orders.csv", customers_path="customers.csv", write=True):
    """Run the orders pipeline end to end. Returns the joined table and a summary."""
    raw = pd.read_csv(orders_path)
    clean_df, rejected_df = clean(raw)
    enriched = enrich(clean_df)
    customers = pd.read_csv(customers_path, parse_dates=["signup_date"])
    joined = enriched.merge(
        customers, on="customer_id", how="left",
        validate="many_to_one", indicator=True, suffixes=("", "_customer"),
    )
    completed = joined[joined["status"] == "completed"]
    summary = {
        "rows_in": len(raw),
        "rows_kept": len(clean_df),
        "rows_rejected": len(rejected_df),
        "orphans": int((joined["_merge"] == "left_only").sum()),
        "revenue_completed": float(completed["revenue"].sum()),
    }
    if write:
        joined.to_csv("clean_orders.csv", index=False)
    return joined, summary


_, summary = run()
print(summary)

In [ ]:
# Clean up
import os, shutil

for name in ["airtravel.csv", "orders.csv", "customers.csv", "orders_may.csv",
             "clean_orders.csv", "summary_by_country.csv", "rejected_orders.csv"]:
    if os.path.isfile(name):
        os.remove(name)

if os.path.isdir("output"):
    shutil.rmtree("output")

print("cleaned up")